# Step 1 — Extract foundation-model embeddings once (IEEE Access revision)

Runtime → Change runtime type → **L4 GPU**. Run cells top to bottom.

Everything is **resumable**: if Colab disconnects, just run all cells again — finished chunks are skipped.

Uses half precision (fp16), checked to give the same decisions as the original fp32 runs. Measured on T4: ~37 patches/s (~5 h per model). L4 is roughly 2–3x faster.

Speed tests and finished pieces are skipped automatically, so **Runtime → Run all** is always safe.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
%cd /content
import os
if os.path.isdir('GP_ECG'):
    %cd GP_ECG
    !git pull
else:
    !git clone https://github.com/LeenRayess/reliable-cross-dataset-lnm.git GP_ECG
    %cd GP_ECG
!git log -1 --oneline

In [ ]:
# ---- settings ----
HF_TOKEN = ''  # paste from https://huggingface.co/settings/tokens (needs access to UNI2-h and Virchow2)
MODELS = ['uni2h', 'virchow2']   # run both in one session, or one per session
DATA = {
    'pcam_macenko':  '/content/drive/MyDrive/GP_ECG_DATA/preprocessed_macenko_benchmark_style',
    'cam17_macenko': '/content/drive/MyDrive/GP_ECG_DATA/wilds/preprocessed_macenko_benchmark_style',
}
OUT_ROOT = '/content/drive/MyDrive/GP_ECG_REVISION/embeddings'

import os
if not HF_TOKEN:
    raise ValueError('Set HF_TOKEN first')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
os.makedirs(OUT_ROOT, exist_ok=True)

In [ ]:
!pip install -q -U timm h5py huggingface_hub
import torch, timm; print(torch.__version__, timm.__version__)

In [ ]:
# Copy only the x/y .h5 files Drive -> local SSD (much faster reads)
import shutil, glob
LOCAL = {}
for tag, src in DATA.items():
    dst = '/content/local_h5/' + tag
    os.makedirs(dst, exist_ok=True)
    for f in sorted(glob.glob(src + '/*_[xy].h5')):
        out = os.path.join(dst, os.path.basename(f))
        if not os.path.exists(out) or os.path.getsize(out) != os.path.getsize(f):
            print('copying', f); shutil.copy(f, out + '.part'); os.replace(out + '.part', out)
    LOCAL[tag] = dst
    print(tag, os.listdir(dst))

### Speed / memory benchmark (for the execution-time comments). Takes ~5 min per model; skipped automatically if already done on this GPU type.

In [ ]:
for m in MODELS:
    !python revision/scripts/extract_embeddings.py --model {m} --benchmark-only --out-root "{OUT_ROOT}"

### Extract embeddings (train / valid / test of both datasets)

In [ ]:
args = ' '.join('--h5-dir "{}" --dataset-tag {}'.format(LOCAL[t], t) for t in DATA)
for m in MODELS:
    !python revision/scripts/extract_embeddings.py --model {m} {args} --out-root "{OUT_ROOT}" --batch-size 256 --num-workers 2 --precision fp16 --store-dtype float16

### Check: every split should say DONE

In [ ]:
import json
for m in MODELS:
    for t in DATA:
        for s in ['train', 'valid', 'test']:
            p = f'{OUT_ROOT}/{m}/{t}/{s}/done.json'
            print(m, t, s, 'DONE ' + str(json.load(open(p))['n_rows']) if os.path.exists(p) else 'MISSING')